In [0]:
%sql
CREATE OR REPLACE VIEW sandbox.gold.metricas_equipo AS
WITH base AS (
    SELECT *,
        DATEDIFF(fecha_apertura, LAG(fecha_apertura) OVER (
            PARTITION BY equipo_id ORDER BY fecha_apertura
        )) AS dias_desde_ultima_falla
    FROM sandbox.gold.ordenes_enriched
)
SELECT
    equipo_id,
    tipo,
    fabricante,
    region,
    COUNT(orden_id)                                                         AS total_ordenes,
    SUM(CAST(fecha_cierre IS NULL AS INT))                                  AS ordenes_abiertas,
    ROUND(AVG(costo_estimado), 2)                                           AS costo_promedio,
    ROUND(SUM(costo_estimado), 2)                                           AS costo_total,
    ROUND(AVG(tiempo_resolucion), 1)                                            AS duracion_promedio_dias,
    ROUND(SUM(CAST(resolucion = 'exitosa' AS INT)) / COUNT(*) * 100, 1)    AS tasa_resolucion_exitosa_pct,
    COUNT(DISTINCT tecnico_asignado)                                        AS tecnicos_distintos,
    ROUND(AVG(dias_desde_ultima_falla), 1)                                  AS mtbf_dias
FROM base
WHERE orden_abierta = FALSE
GROUP BY equipo_id, tipo, fabricante, region;

In [0]:
%sql
select * from sandbox.gold.metricas_equipo